# 19.2 表征分析与激活引导 (Representation Steering)

> 🕐 预估学习时间：35分钟

除了电路级分析，产业中更常用的是表征级技术：线性探针、对比激活差、激活引导（Activation Steering / CAA）与拒绝方向消融。

本节涵盖：
- 线性探针（Probing）
- Contrastive Activation / Steering Vector
- 推理时干预与强度控制
- 与微调/安全护栏的互补关系


## 1. 线性探针：表征里有没有某概念？

训练一个轻量线性分类器读出隐藏状态，判断模型内部是否线性可分地编码了目标属性（毒性、语言、真实性等）。


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

d_model = 64
n = 400
# Synthetic: class bit is linearly encoded in hidden states with noise
labels = torch.randint(0, 2, (n,))
h = torch.randn(n, d_model) * 0.5
h[:, 0] += labels.float() * 2.0  # concept direction

probe = nn.Linear(d_model, 1)
opt = torch.optim.Adam(probe.parameters(), lr=1e-2)

print('=== Linear Probe ===')
for step in range(60):
    logit = probe(h).squeeze(-1)
    loss = F.binary_cross_entropy_with_logits(logit, labels.float())
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 20 == 0 or step == 59:
        acc = ((logit > 0).long() == labels).float().mean().item()
        print(f'step={step:02d} loss={loss.item():.4f} acc={acc:.3f}')

direction = probe.weight.detach().squeeze(0)
direction = direction / (direction.norm() + 1e-8)
print(f'\nProbe direction top dims: {direction.abs().topk(3).indices.tolist()}')
print(f'Key: High probe accuracy means the concept is linearly readable from activations.')


## 2. Steering Vector：对比激活差

**Contrastive Activation Addition (CAA)**：
1. 构造正/负提示对（如"诚实回答" vs "含糊其辞"）
2. 取中间层平均激活差作为引导向量 `v`
3. 推理时 `h ← h + α · v`

优点：无需训练、可即时开关；缺点：强度过大可能损伤流畅性。


In [ ]:
class ToyDecoder(nn.Module):
    def __init__(self, d=64, vocab=50):
        super().__init__()
        self.embed = nn.Embedding(vocab, d)
        self.blocks = nn.ModuleList([
            nn.TransformerEncoderLayer(d, 4, 128, batch_first=True) for _ in range(2)
        ])
        self.head = nn.Linear(d, vocab)

    def hidden(self, ids, layer=1):
        x = self.embed(ids)
        for i, blk in enumerate(self.blocks):
            x = blk(x)
            if i == layer:
                return x
        return x

    def forward(self, ids, steer=None, alpha=1.0, layer=1):
        x = self.embed(ids)
        for i, blk in enumerate(self.blocks):
            x = blk(x)
            if steer is not None and i == layer:
                x = x + alpha * steer
        return self.head(x)


model = ToyDecoder()
pos = torch.randint(0, 50, (32, 10))
neg = torch.randint(0, 50, (32, 10))

with torch.no_grad():
    h_pos = model.hidden(pos).mean(dim=(0, 1))
    h_neg = model.hidden(neg).mean(dim=(0, 1))
    steer = h_pos - h_neg
    steer = steer / (steer.norm() + 1e-8)

probe_ids = torch.randint(0, 50, (8, 10))
base_logits = model(probe_ids)
steered_logits = model(probe_ids, steer=steer, alpha=2.0)

shift = (steered_logits - base_logits).abs().mean().item()
print('=== Activation Steering ===')
print(f'Steering vector norm={steer.norm().item():.4f}')
print(f'Mean |logit shift| at alpha=2.0: {shift:.4f}')

for alpha in [0.0, 0.5, 1.0, 2.0, 4.0]:
    out = model(probe_ids, steer=steer, alpha=alpha)
    entropy = (-F.softmax(out[:, -1], dim=-1) * F.log_softmax(out[:, -1], dim=-1)).sum(-1).mean()
    print(f'alpha={alpha:.1f}: last-token entropy={entropy.item():.3f}')

print(f'\nKey: Steering adds a concept direction at inference time; tune alpha to trade control vs fluency.')


## 3. 拒绝方向消融与互补策略

对安全场景，可估计"拒绝方向"并在越狱时增强它，或在过度拒答时减弱它。

| 方法 | 改参数？ | 可逆？ | 适用 |
|------|---------|-------|------|
| Steering | 否 | 是 | 快速行为调节 |
| LoRA 安全微调 | 是 | 部分 | 长期对齐 |
| Guardrail 外挂 | 否 | 是 | 生产兜底 |
| SAE 特征钳制 | 否 | 是 | 精细概念控制 |

实践中通常 **Steering/护栏做运行时控制，微调做分布偏移**。


In [ ]:
def refusal_score(logits, refuse_token=1, comply_token=2):
    return (logits[:, -1, refuse_token] - logits[:, -1, comply_token]).mean().item()


refusal_dir = steer  # reuse contrastive vector as a stand-in
print('=== Refusal Direction Control ===')
for alpha in [-2.0, -1.0, 0.0, 1.0, 2.0]:
    logits = model(probe_ids, steer=refusal_dir, alpha=alpha)
    print(f'alpha={alpha:+.1f}: refusal_score={refusal_score(logits):+.4f}')

print(f'\nKey: Same vector can increase or decrease a behavior by flipping the sign of alpha.')
print(f'Combine with external guardrails for production defense-in-depth.')


## 课后思考题

1. 探针准确率高是否意味着该概念被模型“用于”决策？如何用因果方法验证？
2. Steering 与 LoRA 安全微调各适合什么变更频率与风险等级？
3. alpha 过大时会出现哪些失败模式？如何自动选择强度？
4. 如何把 SAE 特征与 Steering Vector 结合，做成可开关的概念级控件？

---
> 本节涵盖了19.2 表征分析与激活引导的核心概念与代码实现。建议结合实际项目需求，选择合适的技术方案，并通过实验验证不同方法的效果差异。
